# 05 — VAR Baseline

## Objetivo

Estimar o modelo VAR de referência (baseline) para análise dos efeitos da 
política monetária sobre PIB (proxy: IBC-Br) e inflação (IPCA) no Brasil, 
no período 2003-2025 em frequência mensal.

Este é o coração do projeto. Os resultados deste notebook serão comparados 
com:

- A monografia original (modelo VAR trimestral 2002-2024)
- O BVAR com Minnesota prior (notebook 06)
- O VAR com sign restrictions (notebook 07)

## Decisões herdadas dos blocos anteriores

| Item | Decisão | Origem |
|------|---------|--------|
| Janela temporal | 2003-01 a 2025-12 (276 obs mensais) | Bloco 3 |
| Tratamento da pandemia | Dummies para mar-set 2020 | Bloco 3 |
| Especificação | VAR em diferenças (não VECM) | Bloco 4 |
| Ordem de integração das séries | 8 séries I(1), IPCA I(0) | Bloco 4 |
| Dummies de quebra estrutural | Selic 2017, cambio/indústria 2014, crédito 2015 | Bloco 4 |

## Estrutura do notebook

1. Construção das variáveis finais (diferenças)
2. Construção das dummies de quebra estrutural
3. Definição do conjunto de variáveis
4. Ordenação de Cholesky
5. Seleção de defasagens
6. Estimação e diagnósticos
7. IRFs e decomposição da variância
8. Comparação com a monografia

## 1. Construção das variáveis finais

### Decisão de modelagem

Conjunto final de 6 variáveis para o VAR baseline:

| Variável | Transformação | Papel no modelo |
|----------|--------------|-----------------|
| `ln_commodities` | Primeira diferença | Controle exógeno externo |
| `ln_ibcbr` | Primeira diferença | Atividade econômica |
| `ipca` | Nível | Inflação realizada (I(0)) |
| `exp_ipca_12m` | Primeira diferença | Expectativa forward-looking |
| `selic` | Primeira diferença | Instrumento de política monetária |
| `ln_cambio` | Primeira diferença | Variável cambial |

**Diferenças vs nível:** todas as séries I(1) entram em primeira diferença 
(ver Bloco 4 para fundamentação). O IPCA, classificado como I(0), entra 
em nível.

**Interpretação econômica das diferenças do log:** $\Delta \ln(y_t) = 
\ln(y_t) - \ln(y_{t-1}) \approx$ taxa de crescimento mensal de $y_t$ 
(para valores pequenos). Assim, "$\Delta$ ln_ibcbr" representa 
aproximadamente a taxa de crescimento mensal do indicador de atividade.

### Variáveis fora do baseline (análises de robustez futuras)

- `ln_prod_industrial`: redundante com IBC-Br
- `ln_credito_total`: canal de crédito específico (modelo de robustez)
- `ln_m1`: endógeno sob regime de metas com taxa de juros como instrumento

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PROCESSED = Path('../data/processed')

# Carrega o dataset tratado do Bloco 3
df = pd.read_csv(
    DATA_PROCESSED / 'series_tratadas.csv',
    index_col=0,
    parse_dates=True
)

print(f"Dataset carregado: {df.shape}")
print(f"Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")

Dataset carregado: (276, 16)
Período: 2003-01 a 2025-12


In [6]:
# Construção das variáveis finais do modelo

# Séries que entram em primeira diferença (todas as I(1))
series_diff = [
    'ln_commodities',
    'ln_ibcbr',
    'exp_ipca_12m',
    'selic',
    'ln_cambio',
]

# Séries que entram em nível (I(0))
series_nivel = [
    'ipca',
]

# Constrói o DataFrame do modelo
df_modelo = pd.DataFrame(index=df.index)

# Primeiras diferenças (prefixo "d_" para deixar claro)
for col in series_diff:
    df_modelo[f'd_{col}'] = df[col].diff()

# Séries em nível mantêm o nome original
for col in series_nivel:
    df_modelo[col] = df[col]

# Adiciona as dummies de pandemia (que já existem no df)
dummies_pandemia = [c for c in df.columns if c.startswith('dummy_covid_')]
for col in dummies_pandemia:
    df_modelo[col] = df[col]

print(f"\nDataFrame do modelo:")
print(f"Shape: {df_modelo.shape}")
print(f"\nColunas:")
for c in df_modelo.columns:
    print(f"  - {c}")


DataFrame do modelo:
Shape: (276, 13)

Colunas:
  - d_ln_commodities
  - d_ln_ibcbr
  - d_exp_ipca_12m
  - d_selic
  - d_ln_cambio
  - ipca
  - dummy_covid_2020_03
  - dummy_covid_2020_04
  - dummy_covid_2020_05
  - dummy_covid_2020_06
  - dummy_covid_2020_07
  - dummy_covid_2020_08
  - dummy_covid_2020_09
